In [2]:
import os
import sys
sys.path.insert(0, '')
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import torch
import numpy as np
from PIL import Image
from model import AdaptModel, data_transform, inverse_data_transform
from torchvision import transforms
from matplotlib import pyplot as plt
from tqdm import tqdm
import gc
import pdb
from thop import profile

rank = 'cuda:0'
model = AdaptModel(rank=rank, num_class=9, pretrained='./pretrained/MKF/model.pth')
model.FusionModule.eval()
print('done')

[Info]: Loading config from [./pretrained/IKR/config.yaml] as yaml file.
[Info]: Loading config from [./pretrained/CKG/config.yaml] as yaml file.
==> load pretrained model
done


In [3]:
skip = model.num_timesteps // 25
seq = range(0, model.num_timesteps, skip)
seq_next = [-1] + list(seq[:-1])

In [4]:
color_map = {
    0: [0, 0, 0],       # unlabelled
    1: [64, 0, 128],    # car
    2: [64, 64, 0],     # person
    3: [0, 128, 192],   # bike
    4: [0, 0, 192],     # curve
    5: [128, 128, 0],   # car_stop
    6: [64, 64, 128],   # guardrail
    7: [192, 128, 128], # color_cone
    8: [192, 64, 0],     # bump
}
def visualize_segmentation(label_map):
    h, w = label_map.shape
    colored = np.zeros((h, w, 3), dtype=np.uint8)
    for class_id, color in color_map.items():
        colored[label_map == class_id] = color
    return colored

In [5]:
datapath = './data/vis'
filename = 'cologne_000079_000019_leftImg8bit.png'

In [ ]:
seed = 55
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
# clear
torch.cuda.empty_cache() 
gc.collect()

image = Image.open(os.path.join(datapath, filename)).convert('RGB')
ori_w, ori_h = image.size  
image = image.resize([1184,604]) # max=1184, 604
img = transforms.ToTensor()(image).unsqueeze(0).to(rank)

model.ae.encoder = model.ae.encoder.to(rank) # load
raw_z = model.ae.encoder(img)
raw_z = data_transform(raw_z) 
model.ae.encoder = model.ae.encoder.to('cpu')
torch.cuda.empty_cache() 
gc.collect()

ikr_xt = ckg_xt = fuse_xt = torch.randn_like(raw_z, dtype=torch.float32).cpu()  
batch, c, h, w = raw_z.shape
seg = torch.randn(batch, 9, h, w).cpu() 


# First Stage IKR Module
print("Stage 1: Running IKR module...")
model.IKRModule = model.IKRModule.to(rank)
ikr_out = {} 
ikr_xt = ikr_xt.to(rank)
with torch.no_grad():
    for it, (i, j) in tqdm(enumerate(zip(reversed(seq), reversed(seq_next)))):
        t = (torch.ones(batch) * i).to(rank)
        next_t = (torch.ones(batch) * j).to(rank)
        ikr_noise, ikr_xt, ikr_x0 = model.ikr_sample_onestep(raw_z.float(), ikr_xt.float(), t, next_t)
        ikr_out[it] = {
            'x0': ikr_x0.detach().cpu().clone(),
            'noise': ikr_noise.detach().cpu().clone(),
        }
del ikr_noise, ikr_xt, ikr_x0, t, next_t
model.IKRModule = model.IKRModule.cpu()
torch.cuda.empty_cache() 
gc.collect()

# Second Stage CKG Module
print("Stage 2: Running CKG Module...")
model.CKGModule = model.CKGModule.to(rank)
ckg_out = {} 
ckg_xt = ckg_xt.to(rank)  
with torch.no_grad():
    for it, (i, j) in tqdm(enumerate(zip(reversed(seq), reversed(seq_next)))):
        t = (torch.ones(batch) * i).to(rank)
        next_t = (torch.ones(batch) * j).to(rank)
        ckg_noise, ckg_xt, ckg_x0 = model.ckg_sample_onestep(raw_z.float(), ckg_xt.float(), t, next_t)
        ckg_out[it] = {
            'x0': ckg_x0.detach().cpu().clone(),
            'noise': ckg_noise.detach().cpu().clone(),
        }
del ckg_noise, ckg_xt, ckg_x0, t, next_t,raw_z
model.CKGModule = model.CKGModule.cpu()
torch.cuda.empty_cache() 
gc.collect()

# Final Stage MKF Module
print("Stage 3: Running MKF Module...")
model.FusionModule = model.FusionModule.to(rank)  
# 预热
fusion_input = torch.randn(1,24,32,32).to(rank)
ikr_noise = ckg_noise = torch.randn(1,8,32,32).to(rank)
seg_temp = torch.randn(1,9,32,32).to(rank)
t = torch.ones(1).to(rank)
noise, seg_map, _ = model.FusionModule(fusion_input, ikr_noise, ckg_noise, t, seg_temp)
del fusion_input, ikr_noise, ckg_noise, seg_temp, noise, seg_map, t
fuse_xt = fuse_xt.to(rank)
seg = seg.to(rank)
with torch.no_grad():
    for it, (i, j) in tqdm(enumerate(zip(reversed(seq), reversed(seq_next))), desc="Fusion"):
        t = (torch.ones(batch) * i).to(rank)
        next_t = (torch.ones(batch) * j).to(rank)

        ikr_x0 = ikr_out[it]['x0'].to(rank)
        ckg_x0 = ckg_out[it]['x0'].to(rank)
        ikr_noise = ikr_out[it]['noise'].to(rank)
        ckg_noise = ckg_out[it]['noise'].to(rank)
        at = model.alphas_cumprod_pre.index_select(0, t.long()+1).view(-1, 1, 1, 1)
        at_next = model.alphas_cumprod_pre.index_select(0, next_t.long()+1).view(-1, 1, 1, 1)
        
        fusion_input = torch.cat([ikr_x0, ckg_x0, fuse_xt], dim=1)
        noise, seg_map, seg = model.FusionModule(fusion_input, ikr_noise, ckg_noise, t, seg)
        x0 = (fuse_xt - noise * (1 - at).sqrt()) / at.sqrt()
        c2 = (1 - at_next).sqrt()
        fuse_xt = at_next.sqrt() * x0 + c2 * noise
del seg, fuse_xt,  t, next_t, ikr_out, ckg_out
model.FusionModule = model.FusionModule.cpu()
torch.cuda.empty_cache() 
gc.collect()

print("Decoding final images...")
model.ae.decoder = model.ae.decoder.to(rank)

ikrz = inverse_data_transform(ikr_x0)
ckgz = inverse_data_transform(ckg_x0)
imagez = inverse_data_transform(x0)

ikrimg = model.ae.decoder(ikrz.float())
ckgimg = model.ae.decoder(ckgz.float())
MagImg = model.ae.decoder(imagez.float())
del ikrz, ckgz, imagez
model.ae.decoder = model.ae.decoder.cpu()
torch.cuda.empty_cache() 
gc.collect()

ikrimg=ikrimg.squeeze(0).permute(1, 2, 0)
ikrimg=ikrimg.cpu().numpy()

ckgimg=ckgimg.squeeze(0).permute(1, 2, 0)
ckgimg=ckgimg.cpu().numpy()

MagImg=MagImg.squeeze(0).permute(1, 2, 0)
MagImg=MagImg.cpu().numpy()

seg_map = torch.softmax(seg_map, dim=1)
seg_label = torch.argmax(seg_map, dim=1)
segimg=seg_label.squeeze(0).cpu().numpy()
segimg=visualize_segmentation(segimg)


# show result
_,ax = plt.subplots(2,3,figsize=(8,6))
plt.rcParams['axes.titlesize']=10
plt.subplots_adjust(hspace=0.2, wspace=0)
[a.axis('off') for a in ax.flatten()]
ax[0,0].imshow(image)
ax[0,0].set_title('raw')
ax[0,1].imshow(ikrimg)
ax[0,1].set_title('ikr')
ax[0,2].imshow(ckgimg)
ax[0,2].set_title('ckg')
ax[1,0].imshow(MagImg)
ax[1,0].set_title('MagImg')
ax[1,1].imshow(segimg)
ax[1,1].set_title('seg')
plt.show()

# save result
save_path = os.path.join('results')
os.makedirs(save_path, exist_ok=True)

MagImg = np.clip(MagImg * 255, 0, 255).astype(np.uint8)
MagImg = Image.fromarray(MagImg, mode='RGB')
MagImg = MagImg.resize([ori_w, ori_h])
MagImg.save(os.path.join(save_path, 'MagImg_'+filename)) 

segimg = Image.fromarray(segimg, mode='RGB')
segimg = segimg.resize([ori_w, ori_h])
segimg.save(os.path.join(save_path, 'seg_'+filename)) 

ikrimg = np.clip(ikrimg * 255, 0, 255).astype(np.uint8) 
ikrimg = Image.fromarray(ikrimg, mode='RGB')
ikrimg = ikrimg.resize([ori_w, ori_h])
ikrimg.save(os.path.join(save_path, 'ikr_'+filename)) 

ckgimg = np.clip(ckgimg * 255, 0, 255).astype(np.uint8) 
ckgimg = Image.fromarray(ckgimg, mode='RGB')
ckgimg = ckgimg.resize([ori_w, ori_h])
ckgimg.save(os.path.join(save_path, 'ir_'+filename)) 
print(f'image saved to {save_path}')

del ikrimg, ckgimg, MagImg, seg_map, seg_label, segimg, img
torch.cuda.empty_cache() # clear
gc.collect()